In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-small"
).to(device)

In [4]:
checkpoint_path = "whisper-small-hi/checkpoint-100"

model = PeftModel.from_pretrained(base_model, checkpoint_path)

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.


In [5]:
processor = AutoProcessor.from_pretrained("openai/whisper-small")

In [6]:
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="hi",
    task="transcribe"
)

In [9]:
from datasets import load_dataset, Audio

dataset = load_dataset(
    "google/fleurs",
    "hi_in",
    split="test[:1]",
    trust_remote_code=True   # 👈 Add this line
)

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
sample = dataset[0]
audio = sample["audio"]["array"]

In [10]:

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt",
    padding="max_length",
    truncation=True
)

input_features = inputs.input_features.to(device)

In [11]:
with torch.no_grad():
    predicted_ids = model.generate(input_features)

prediction = processor.decode(predicted_ids[0], skip_special_tokens=True)

print("Prediction:", prediction)
print("Reference :", sample["transcription"])

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prediction: 0.03,  13.7,  21434,  कुछ अड़्वो में अज्ट्र केंडरक होता है जिसका मतलब यहां की उन्में थोड़े या बिना किसी जटके से तूटने की प्रवत्ती होती है,  14.6,  25.7,  21434,  अज्ट्र केंडरक होता है जिसका मतलब यहां की उन्में थोड़े या बिना किसी ज�
Reference : कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है


In [12]:
model = model.merge_and_unload()

model.save_pretrained("final_model")
processor.save_pretrained("final_model")

c:\Users\jayes\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


[]

In [14]:
from transformers import AutoModelForSpeechSeq2Seq
from peft import PeftModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(checkpoint_path):
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        "openai/whisper-small"
    ).to(device)

    model = PeftModel.from_pretrained(base_model, checkpoint_path)

    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="hi",
        task="transcribe"
    )

    return model

In [15]:
from transformers import AutoModelForSpeechSeq2Seq
from peft import PeftModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_model(checkpoint_path):
    base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
        "openai/whisper-small"
    ).to(device)

    model = PeftModel.from_pretrained(base_model, checkpoint_path)

    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="hi",
        task="transcribe"
    )

    return model

In [17]:
from datasets import load_dataset, Audio

fleurs_test = load_dataset(
    "google/fleurs",
    "hi_in",
    split="test[:100]"   # ⚡ small subset for speed
)

fleurs_test = fleurs_test.cast_column("audio", Audio(sampling_rate=16000))

In [18]:
import os

base_path = "whisper-small-hi"

checkpoints = sorted([
    os.path.join(base_path, d)
    for d in os.listdir(base_path)
    if d.startswith("checkpoint-")
])

print(checkpoints)

['whisper-small-hi\\checkpoint-100', 'whisper-small-hi\\checkpoint-1000', 'whisper-small-hi\\checkpoint-200', 'whisper-small-hi\\checkpoint-300', 'whisper-small-hi\\checkpoint-400', 'whisper-small-hi\\checkpoint-500', 'whisper-small-hi\\checkpoint-600', 'whisper-small-hi\\checkpoint-700', 'whisper-small-hi\\checkpoint-800', 'whisper-small-hi\\checkpoint-900']


In [20]:
import evaluate
import torch

wer_metric = evaluate.load("wer")

def compute_wer(model, dataset):
    predictions = []
    references = []

    for sample in dataset:
        audio = sample["audio"]["array"]

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )

        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            pred_ids = model.generate(input_features)

        pred_text = processor.decode(pred_ids[0], skip_special_tokens=True)

        predictions.append(pred_text)
        references.append(sample["transcription"])

    wer = 100 * wer_metric.compute(
        predictions=predictions,
        references=references
    )

    return wer

In [ ]:
results = {}

for ckpt in checkpoints:
    print(f"\n🔍 Evaluating {ckpt}")

    model = load_model(ckpt)

    wer = compute_wer(model, fleurs_test)

    print(f"✅ WER: {wer:.2f}%")

    results[ckpt] = wer


🔍 Evaluating whisper-small-hi\checkpoint-100
✅ WER: 147.03%

🔍 Evaluating whisper-small-hi\checkpoint-1000
✅ WER: 93.94%

🔍 Evaluating whisper-small-hi\checkpoint-200


In [ ]:
best_checkpoint = min(results, key=results.get)

print("\n🏆 BEST CHECKPOINT:")
print(f"{best_checkpoint} → WER: {results[best_checkpoint]:.2f}%")